# Module 1 : identité de l'expéditeur

L'identité affichée correspond-elle à l'expéditeur réel ?

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/secumail")
except ImportError:
    os.chdir(Path.cwd().parents[1])  # depuis workshop/<atelier>/

sys.path.insert(0, os.getcwd())

## Spam ou hameçonnage ?

<img src="images/spam_vs_hameconnage.svg" alt="" style="width:820px">

Un hameçonnage bien rédigé ne contient aucun mot suspect. C'est le **domaine** qui le trahit.

## Le nom affiché, c'est du texte

<img src="images/nom_affiche.svg" alt="" style="width:720px">

## La vérité est après le `@`

In [ ]:
from src.email import extraire_domaine_adresse

expediteur = '"Instagram Support" <instagram.help2024@gmail.com>'
extraire_domaine_adresse(expediteur)

## Un domaine se lit de droite à gauche

Un seul morceau de l'ancienne passerelle a survécu : `domaine_est_officiel()`.
Peut-on lui faire confiance ?

In [ ]:
from homework.phishing_rules import domaine_est_officiel

officiels = ["desjardins.com", "amazon.ca", "microsoft.com"]
for domaine in [
    "accesd.desjardins.com",
    "desjardins.com.securite-client.xyz",
    "amazon.ca",
    "micr0soft.com",
]:
    officiel = domaine_est_officiel(domaine, officiels)
    print(f"{domaine:<36} -> {officiel}")

**Lequel est faux ? Pourquoi `in` ne suffit-il pas ?** (exercice 2.2)

## Une adresse se lit de droite à gauche

<img src="images/domaine_maison.svg" alt="" style="width:760px">

## Comparer la fin, pas le contenu

Le vrai propriétaire est à la **fin** du domaine. Python sait tester la fin d'un texte :

In [ ]:
print("accesd.desjardins.com".endswith("desjardins.com"))
print("desjardins.com.securite-client.xyz".endswith("desjardins.com"))
print("fauxdesjardins.com".endswith("desjardins.com"))

Le troisième est un piège. **Comment le corriger ?**

## Solution

Un sous-domaine officiel finit par **un point** suivi du domaine officiel :

`domaine == officiel or domaine.endswith("." + officiel)`

## Les sosies

<img src="images/sosies.svg" alt="" style="width:600px">

Pour démasquer un sosie, on fait le remplacement inverse.

## Le remplacement inverse

**À vous :** que deviennent ces domaines après le remplacement ?
`g00gle.com`, `vvhatsapp.com`, `app1e.com`, `google.com`

In [ ]:
from homework.phishing_rules import SUBSTITUTIONS

for domaine in ["g00gle.com", "vvhatsapp.com", "app1e.com", "google.com"]:
    corrige = domaine
    for faux, vrai in SUBSTITUTIONS.items():
        corrige = corrige.replace(faux, vrai)
    print(f"{domaine:<16} -> {corrige}")

`g00gle` devient `google` : la marque était cachée, c'est un **sosie**.
`google.com` ne change pas : c'est le **vrai**.

## Un lien inconnu n'est pas suspect en soi

Presque tous les liens sur internet pointent vers des domaines absents de notre liste : un article de presse, une boutique locale, etc.
Une règle qui les signale tous sonne l'alerte tout le temps. Et un détecteur qui signale tout ne détecte rien.

L'hameçonnage, c'est un lien inconnu **qui se fait passer pour** une marque connue.
C'est la **contradiction** qui est l'indice.

<img src="images/contradiction.svg" alt="" style="width:760px">

## Quelle marque le courriel prétend-il représenter ?

In [ ]:
from src.email import extraire_domaine_url, marque_annoncee

marques = ["amazon", "desjardins", "netflix", "instagram"]
faux_netflix = {"name": "Netflix", "subject": "Echec de paiement"}
pub = {"name": "Super Rabais", "subject": "LIQUIDATION TOTALE"}
lien = "http://netflix-billing.tk/update.php"

print("faux_netflix :", marque_annoncee(faux_netflix, marques))
print("pub          :", marque_annoncee(pub, marques))
print("lien         :", extraire_domaine_url(lien))

Le faux Netflix se réclame de `netflix` et son lien mène à `netflix-billing.tk` : **contradiction**.
La pub ne se fait passer pour personne : la règle doit rester **muette**. (exercices 2.4 et 2.5)

## Le contrat : des points et une explication

Chaque règle renvoie un couple `(points, explication)`.
Un système de sécurité qui affirme sans justifier est inutilisable : personne ne peut vérifier sa décision.

Une règle déjà écrite. Elle compare les **domaines** de `From` et de `Reply-To` :

In [ ]:
from src.scoring_engine import regle_reply_to

print(regle_reply_to("facturation@fournisseur.ca", "paiements@fournisseur-inv.top"))
print(regle_reply_to("no-reply@netflix.com", ""))

Votre `regle_return_path()` lui ressemble beaucoup. (exercices 2.9 à 2.11)

## Une lettre, deux adresses

<img src="images/enveloppe_lettre.svg" alt="" style="width:700px">

## Le cas de l'infolettre

Deux adresses différentes, est-ce une fraude ?

In [ ]:
print(extraire_domaine_adresse("info@infolettre-cuisine.com"))
print(extraire_domaine_adresse("bounce-4471@infolettre-cuisine.com"))

Même domaine : c'est le fonctionnement normal des rebonds. Comparez les **domaines**, jamais les adresses complètes.
Et un `Return-Path` vide ne prouve rien. (exercice 2.10)

## Jouez au détecteur

Qui a vraiment écrit ce courriel ? Trouvez les indices qui trahissent l'expéditeur.

<img src="images/detective_question.svg" alt="" style="width:860px">

## Les indices, et les règles qui les attrapent

<img src="images/detective_reponses.svg" alt="" style="width:860px">

Trois de ces règles restent à coder. La quatrième est déjà écrite.

# À vous

Fichier : `homework/phishing_rules.py`

| Fonction | Exercices |
|---|---|
| `domaine_est_officiel` (à corriger) | 2.1, 2.2 |
| `regle_return_path` | 2.9, 2.10 |
| `regle_nom_vs_adresse` | 2.7, 2.8 |
| `regle_lien_incoherent` | 2.4, 2.5 |
| `regle_substitution` | 2.1 |

Après chaque fonction, relancez la section **Vérification** de `main.ipynb`.